# Appendix Notebook A5: Case A for Quantum and Classical Financial Classification

This appendix notebook is the standardised review-paper version of the original local working file for Module 4.

This notebook accompanies Module 4 Case A and compares classical classification baselines against quantum-enhanced alternatives under a common financial feature-engineering pipeline.

## Reading Guide

The notebook moves from data acquisition to feature design, baseline evaluation, quantum model construction, and consolidated result reporting.

All file names in `review_paper/notebooks/` follow a common naming convention so the paper appendix can reference them consistently.

# Case A: Notebook Setup

This notebook standardises the environment and imports used in the Case A comparison.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score

import pennylane as qml

SEED = 42
np.random.seed(SEED)

# -------- User inputs (editable) --------
TICKER = "SPY"
START_DATE = "2018-01-01"
END_DATE   = "2025-12-20"  # per your preference
TEST_RATIO = 0.25          # chronological split

ALPHA_PROBA_THRESHOLD = 0.5  # for Acc/BalAcc/F1

# Quantum model sizes (keep small for NISQ-style)
N_QUBITS = 6
N_LAYERS_QNN = 3

# QSVC kernel compute budget control
MAX_QSVC_TRAIN = 400   # increase if you want (will be slower)
MAX_QSVC_TEST  = 400

print("Config:")
print(" Ticker:", TICKER)
print(" Date range:", START_DATE, "to", END_DATE)
print(" Test ratio:", TEST_RATIO)
print(" Qubits:", N_QUBITS, "| QNN layers:", N_LAYERS_QNN)
print(" QSVC max train/test:", MAX_QSVC_TRAIN, MAX_QSVC_TEST)

## Market Data Acquisition

The selected asset and sample window are downloaded and cleaned before any modelling begins.

In [ ]:
df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)
if df.empty:
    raise ValueError("No data downloaded. Check ticker/date range.")

# Use Close only (auto_adjust=True => adjusted)
# yfinance may return MultiIndex columns for single ticker in some environments
if isinstance(df.columns, pd.MultiIndex):
    # prefer 'Close' (auto_adjust=True already adjusted)
    close = df["Close"].copy()
    # if still a DataFrame (e.g., column level includes ticker), squeeze to Series
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
else:
    close = df["Close"].copy()

df = pd.DataFrame({"Close": close}).dropna()

# Log return
df["r"] = np.log(df["Close"] / df["Close"].shift(1))

# Label: next-day direction
df["y"] = (df["r"].shift(-1) > 0).astype(int)

# -------- Feature engineering (compact + stable) --------
# Lagged returns
for k in [1, 2, 3, 5]:
    df[f"r_lag{k}"] = df["r"].shift(k)

# Rolling volatility proxy (std of returns)
for w in [5, 10, 20]:
    df[f"vol{w}"] = df["r"].rolling(w).std()

# Rolling mean return (momentum proxy)
for w in [5, 10, 20]:
    df[f"mom{w}"] = df["r"].rolling(w).mean()

# Simple price trend proxy: log price vs rolling mean
for w in [10, 20]:
    df[f"trend{w}"] = np.log(df["Close"] / df["Close"].rolling(w).mean())

# Drop NA
df = df.dropna().copy()

FEATURE_COLS = [c for c in df.columns if c.startswith(("r_lag", "vol", "mom", "trend"))]
X = df[FEATURE_COLS].values
y = df["y"].values

print("Data rows:", len(df))
print("Feature dim:", X.shape[1])
print("Feature columns:", FEATURE_COLS)

## Label Construction and Feature Engineering

The target variable and the classical feature space are prepared in a way that can also feed the quantum models.

In [ ]:
n = len(y)
n_test = int(np.floor(TEST_RATIO * n))
n_train = n - n_test

X_train, y_train = X[:n_train], y[:n_train]
X_test,  y_test  = X[n_train:], y[n_train:]

print("Train size:", len(y_train), "| Test size:", len(y_test))
print("Train positive rate:", y_train.mean(), "| Test positive rate:", y_test.mean())

## Evaluation Utilities

Shared evaluation functions are defined so classical and quantum models are compared on the same basis.

In [ ]:
def eval_binary(y_true, y_proba, name="model"):
    y_pred = (y_proba >= ALPHA_PROBA_THRESHOLD).astype(int)
    out = {
        "model": name,
        "AUC": roc_auc_score(y_true, y_proba),
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAcc": balanced_accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred)
    }
    return out

results = []

## Classical Baselines

The notebook first estimates conventional reference models before introducing quantum-enhanced variants.

In [ ]:
# 4.1 Logistic Regression
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=SEED))
])
lr.fit(X_train, y_train)
proba_lr = lr.predict_proba(X_test)[:, 1]
results.append(eval_binary(y_test, proba_lr, "LR"))

# 4.2 SVC with RBF kernel
svc = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", probability=True, random_state=SEED))
])
svc.fit(X_train, y_train)
proba_svc = svc.predict_proba(X_test)[:, 1]
results.append(eval_binary(y_test, proba_svc, "SVC(RBF)"))

# 4.3 Small MLP
mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        max_iter=200,
        random_state=SEED
    ))
])
mlp.fit(X_train, y_train)
proba_mlp = mlp.predict_proba(X_test)[:, 1]
results.append(eval_binary(y_test, proba_mlp, "NN(MLP)"))

pd.DataFrame(results)

## Quantum Linear Model

This section prepares a dimension-reduced and qubit-compatible representation for the linear quantum model.

In [ ]:
# We'll standardize features first, then compress to N_QUBITS dims by simple linear projection.
scaler_q = StandardScaler()
X_train_s = scaler_q.fit_transform(X_train)
X_test_s  = scaler_q.transform(X_test)

# Linear projection to match qubits (keep deterministic)
d_in = X_train_s.shape[1]
W_proj = np.random.RandomState(SEED).normal(size=(d_in, N_QUBITS))
W_proj /= np.linalg.norm(W_proj, axis=0, keepdims=True)  # normalize columns

def project_to_qubits(Xs):
    Z = Xs @ W_proj
    # squash to [-pi, pi] for rotation angles
    Z = np.tanh(Z) * np.pi
    return Z

Xq_train = project_to_qubits(X_train_s)
Xq_test  = project_to_qubits(X_test_s)

print("Quantum feature input shape:", Xq_train.shape, Xq_test.shape)

dev = qml.device("default.qubit", wires=N_QUBITS)

def feature_map(x):
    # simple data-encoding feature map (angle embedding + entanglement)
    qml.AngleEmbedding(x, wires=range(N_QUBITS), rotation="Y")
    for i in range(N_QUBITS - 1):
        qml.CZ(wires=[i, i+1])
    # ring closure (optional)
    qml.CZ(wires=[N_QUBITS-1, 0])

@qml.qnode(dev)
def quantum_features(x):
    feature_map(x)
    # return expectation values as quantum-derived features (dimension N_QUBITS)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

# Precompute quantum-derived features for QLR
QF_train = np.array([quantum_features(x) for x in Xq_train])
QF_test  = np.array([quantum_features(x) for x in Xq_test])

print("Quantum-derived feature shape:", QF_train.shape)

## Logistic Comparison Layer

Results from the quantum and classical logistic-style models are lined up for direct inspection.

In [ ]:
qlr = LogisticRegression(max_iter=2000, random_state=SEED)
qlr.fit(QF_train, y_train)
proba_qlr = qlr.predict_proba(QF_test)[:, 1]
results.append(eval_binary(y_test, proba_qlr, "QLR(quantum-features)"))

pd.DataFrame(results)

## Quantum Kernel Construction

A quantum feature map is used to build kernel-based classifiers in Hilbert space.

In [ ]:
# Quantum kernel: K(x_i, x_j) = |<phi(x_i)|phi(x_j)>|^2
dev_k = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_k)
def state_phi(x):
    feature_map(x)
    return qml.state()

def qkernel(x1, x2):
    s1 = state_phi(x1)
    s2 = state_phi(x2)
    return np.abs(np.vdot(s1, s2))**2

# Subsample for tractability
ntr = min(len(Xq_train), MAX_QSVC_TRAIN)
nte = min(len(Xq_test),  MAX_QSVC_TEST)

Xq_tr_sub = Xq_train[:ntr]
y_tr_sub  = y_train[:ntr]
Xq_te_sub = Xq_test[:nte]
y_te_sub  = y_test[:nte]

print("QSVC subset sizes:", ntr, nte)

# Build Gram matrices (with progress bars)
K_train = np.zeros((ntr, ntr))
print("Building QSVC training Gram matrix...")
for i in tqdm(range(ntr), desc="QSVC train rows"):
    for j in range(i, ntr):
        val = qkernel(Xq_tr_sub[i], Xq_tr_sub[j])
        K_train[i, j] = val
        K_train[j, i] = val

K_test = np.zeros((nte, ntr))
print("Building QSVC test Gram matrix...")
for i in tqdm(range(nte), desc="QSVC test rows"):
    for j in range(ntr):
        K_test[i, j] = qkernel(Xq_te_sub[i], Xq_tr_sub[j])

qsvc = SVC(kernel="precomputed", probability=True, random_state=SEED)
qsvc.fit(K_train, y_tr_sub)

proba_qsvc = qsvc.predict_proba(K_test)[:, 1]
results.append(eval_binary(y_te_sub, proba_qsvc, "QSVC(quantum-kernel)"))

pd.DataFrame(results)

## Quantum Neural Network Layer

A variational model is trained on the same task to compare expressive but resource-limited quantum learning.

In [ ]:
dev_qnn = qml.device("default.qubit", wires=N_QUBITS)
qnp = qml.numpy

def variational_block(theta):
    for l in range(N_LAYERS_QNN):
        for i in range(N_QUBITS):
            qml.Rot(theta[l, i, 0], theta[l, i, 1], theta[l, i, 2], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[N_QUBITS-1, 0])

@qml.qnode(dev_qnn)
def qnn_forward(x, theta):
    feature_map(x)
    variational_block(theta)
    return qml.expval(qml.PauliZ(0))

def sigmoid(z):
    return 1.0 / (1.0 + qnp.exp(-z))

def bce_loss(y_true, y_prob, eps=1e-9):
    y_prob = qnp.clip(y_prob, eps, 1 - eps)
    return -(y_true * qnp.log(y_prob) + (1 - y_true) * qnp.log(1 - y_prob))

# Train subset
N_QNN_TRAIN = min(800, len(Xq_train))
N_QNN_TEST  = min(800, len(Xq_test))
Xq_tr = Xq_train[:N_QNN_TRAIN]
y_tr  = y_train[:N_QNN_TRAIN]
Xq_te = Xq_test[:N_QNN_TEST]
y_te  = y_test[:N_QNN_TEST]

# init
rng = np.random.RandomState(SEED)
theta0 = 0.05 * rng.normal(size=(N_LAYERS_QNN, N_QUBITS, 3))
theta_pl = qnp.array(theta0, requires_grad=True)

LR_QNN = 0.2
EPOCHS = 30
SCALE_LOGIT = 2.0

def objective(theta_var):
    # NO inner tqdm here -> avoids many progress bars
    logits = qnp.stack([qnn_forward(x, theta_var) for x in Xq_tr])
    probs  = sigmoid(SCALE_LOGIT * logits)
    return qnp.mean(bce_loss(y_tr, probs))

opt = qml.GradientDescentOptimizer(stepsize=LR_QNN)

for ep in tqdm(range(1, EPOCHS + 1), desc="QNN training (epochs)"):
    theta_pl, loss_val = opt.step_and_cost(objective, theta_pl)
    if ep == 1 or ep % 5 == 0:
        print(f"Epoch {ep:02d} | loss = {float(loss_val):.5f}")

# Evaluate (keep ONE more progress bar)
logits_te = []
for x in tqdm(Xq_te, desc="QNN inference (test)"):
    logits_te.append(qnn_forward(x, theta_pl))
logits_te = np.array(logits_te, dtype=float)

proba_qnn = 1.0 / (1.0 + np.exp(-SCALE_LOGIT * logits_te))  # numpy ok here
results.append(eval_binary(y_te, proba_qnn, "QNN(variational)"))

pd.DataFrame(results)

## Consolidated Result Table

All model outputs are gathered into a final comparison frame for interpretation.

In [ ]:
res_df = pd.DataFrame(results)
res_df = res_df.sort_values("AUC", ascending=False).reset_index(drop=True)
res_df

## Interpretation Note

Case A is retained as an appendix notebook so readers can inspect modelling details without interrupting the conceptual flow of the main paper.